# Case study 2 · scenario 1 · calibration from available data

Section 4.2 of the MIDDoE paper: Tabrizi Z., Barbera E., Leal da Silva W.R., Bezzo F. (2025). *MIDDoE: An MBDoE Python package for model identification, discrimination, and calibration.* Digital Chemical Engineering 17, 100276. https://doi.org/10.1016/j.dche.2025.100276

**The process.** A pharmaceutical batch reaction, starting material (SM) + TMA → product, with an impurity.
Three concentrations are measured [mol/L]: `y1` starting material, `y2` product, `y3` impurity. The controls
are the temperature profile `u1(t)` [K] and the initial concentrations of SM (`w1`, key `y10`) and TMA
(`w2`, key `y20`). The model (`model.py`, paper eq. 20) has six parameters, many of them hard to estimate
from few data.

**Scenario 1.** Four experiments already exist and no new one can be run. Data are added one experiment at a
time; after each, the estimability analysis decides which parameters the data can support, and only those are
estimated (paper Table 8). The final model is cross-validated.

| Step | What happens | MIDDoE call |
|---|---|---|
| 1 | Describe the controls, measurements and model | `system`, `models`, `insilicos` |
| 2–3 | Add one experiment; fit; estimability analysis; select and estimate the subset (×4) | `expera`, `parmest`, `uncert`, `estima` |
| 4 | Cross-validate the final model | `iden_valida.validation` |

Paper: θ3 ranks first in every round; with all the data the ranking is θ3, θ4, θ5, θ6, θ1, θ2 and the first
three are estimable and pass the t-test. Validation R² = 0.9985 ± 0.0015.

**How to run.** Open this notebook from its own folder (Jupyter and PyCharm do this by default) and run all
cells in order. MIDDoE writes every output next to the notebook: `data.xlsx` (the in-silico experiments, one
sheet per experiment), `iden_results.jac` (all rounds) and the folders `modelling/`, `confidence/`, `design/`,
`post_processing/`, `estimability/`, `validation/`. `krnl_expera` *appends* to `data.xlsx`, so delete the outputs before running again.
The in-silico noise is random, so the numbers differ slightly from the paper on every run; the conclusions
should not.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/middoe_paper/case_study_2_model_calibration/scenario_1_available_data/runner.ipynb)

**On Google Colab?** Run the next cell first: it installs MIDDoE and downloads this case's model file(s)
into the session (on your own PC it does nothing). Colab gives 2 CPU cores and ends idle sessions, so for a
first run lower `num_parallel_runs`, the design iterations and `nboot`.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s) of this case; does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    base = 'https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/middoe_paper/case_study_2_model_calibration/scenario_1_available_data/'
    for f in ['model.py']:
        urllib.request.urlretrieve(base + f, f)

## 1 · Parameters

In [ ]:
from pathlib import Path

theta_true  = [50000, 75000, 0.4116, 111900, 9905, 30000]   # the in-silico process
theta_guess = [100000, 100000, 1, 100000, 100, 10000]       # starting values (Table 7)
theta_min   = [10000, 0, 0.1, 50000, 10, 10000]             # bounds (Table 7)
theta_max   = [1000000, 200000, 10, 200000, 10000, 200000]

## 2 · The `system` dictionary: design space and constraints (paper Table 6)

- `u1`, the temperature, is a profile of 6 levels joined by linear ramps (`'LPF'`) that may only rise
  (`'inc'`), with at least 0.01 K between levels and 0.3 h between switches.
- Each output is sampled 17 times, at least 0.3 h apart, always at 0 and 16 h (`samp_f`), and all three
  together (same `samp_s` label: "synchronised").
- `unc` is the noise standard deviation: 0.005 mol/L absolute ("0.5 % absolute normal noise", Section 4.2.2).
- `w1` and `w2` may range from 0.3 and 0.19 to 1 mol/L. Table 6 prints 0.05 and 0.1 as lower bounds; the
  runs behind the paper used 0.3 and 0.19, kept here.
- `t_d` keeps switches and free samples 0.3 h away from the start and the end.

In [ ]:
sample = {'init': 0, 'meas': True,  # every output: measured, 17 synchronised samples, 0.3 h apart,
          'sp': 17, 'offt': 0.3,    # always at 0 and 16 h, noise sd 0.005 mol/L
          'samp_s': 1, 'samp_f': [0, 16], 'unc': 0.005}

system = {
    'tvi': {'u1': {                       # temperature [K]
        'stps': 6, 'cvp': 'LPF', 'const': 'inc',   # 6 levels, linear ramps, rising only
        'min': 296.15, 'max': 306.15,
        'offl': 0.01, 'offt': 0.3,                 # min level change [K], min time between switches [h]
    }},
    'tvo': {'y1': dict(sample), 'y2': dict(sample), 'y3': dict(sample)},  # SM, product, impurity [mol/L]
    'tii': {'y10': {'min': 0.3, 'max': 1},         # w1, initial SM [mol/L]
            'y20': {'min': 0.19, 'max': 1}},       # w2, initial TMA [mol/L]
    'tio': {},
    't_s': [0, 16],      # 16-h batch
    't_r': 0.02,         # time grid [h]
    't_d': 0.3,          # dead time at both ends [h]
}

## 3 · The `models` and `insilicos` dictionaries

`insilicos` lists the four available experiments (Section 4.2.2): constant temperature at 296.15 or
306.15 K, at two sets of initial concentrations.

In [ ]:
models = {
    'can_m': ['M'],                              # one model: calibration, not discrimination
    'krt': {'M': 'pys'},                         # standalone Python file
    'src': {'M': str(Path('model.py').resolve())},
    'theta': {'M': theta_guess},
    't_l': {'M': theta_min},
    't_u': {'M': theta_max},
    'mutation': {'M': [True] * 6},               # True = estimated, False = held at its current value
}

insilicos = {
    'tr_m': 'M', 'theta': theta_true,
    'errt': 'abs',                     # absolute noise: sigma = unc
    'prels': {
        '1': {'u1': 296.15, 'y10': 0.366, 'y20': 0.19},
        '2': {'u1': 306.15, 'y10': 0.366, 'y20': 0.19},
        '3': {'u1': 296.15, 'y10': 0.65, 'y20': 0.595},
        '4': {'u1': 306.15, 'y10': 0.65, 'y20': 0.595},
    },
}

## 4 · The `iden_opt` dictionary: estimation, uncertainty and estimability

Weighted least squares with L-BFGS-B (`'LMBFGS'`, called `'LBFGSB'` when the paper was run) from several
random starts (`ms`), as the paper states for the estimability analysis. Because several parameters are
poorly identifiable, the covariance comes from 500 bootstrap replicates (`'B'`) rather than from the Fisher
matrix. The tolerance is tighter than in the paper-era runs (0.1), which left θ5 unconverged.

In [ ]:
iden_opt = {
    'meth': 'LMBFGS',     # L-BFGS-B
    'ms': True,           # multi-start: random starts within the bounds, best fit kept
    'maxit': 500, 'tol': 1e-8,
    'ob': 'WLS',          # weighted least squares
    'var-cov': 'B',       # covariance by bootstrap
    'nboot': 500,         # bootstrap replicates
    'sens_m': 'central',  # central finite differences for the sensitivities (also used by the EA)
    'eps': 1e-5,          # their step, in the normalised parameter space
    'c_plt': False,       # no confidence-ellipse plots
    'f_plt': True,        # plot fits against the data
    'plt_s': True,        # show the plots here as well as saving them
    'log': False,
}

## 5 · Four rounds: add an experiment, then identify (Table 8)

`expera(..., expr=r)` adds experiment *r* of `prels` to `data.xlsx`. Each round runs four calls on all the data so far:

1. `parmest(..., case='strov')` fits the model, always starting from the original guess;
2. `uncert` computes the bootstrap covariance, confidence intervals, t-values and R²;
3. `estima` is the **estimability analysis**: it ranks the parameters by orthogonalising the scaled
   sensitivity matrix and keeps the top *k* chosen by the corrected critical ratio rCC (Wu et al., 2011).
   It then fixes the others in `models['mutation']` and stores the subset's estimates in `models['theta']`;
4. `save_rounds` stores the round with its estimability results.

All six parameters are freed before each fit, so every round re-ranks all of them (this is what the
paper-era `parmest` did by itself).

In [ ]:
from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.sc_estima import estima
from middoe.log_utils import save_rounds

round_data = {}
for r in (1, 2, 3, 4):
    expera(system, models, insilicos, design_decisions={}, expr=r)
    label = f'available data, experiments 1-{r}'
    models['mutation']['M'] = [True] * 6                      # re-rank all six parameters
    resultpr = parmest(system, models, iden_opt, case='strov')
    unc = uncert(resultpr, system, models, iden_opt)
    ranking, k_opt, rCC, J_k, best = estima(unc['results'], system, models, iden_opt, r)
    save_rounds(r, unc['results'], label, round_data, models, iden_opt, unc['obs'], system,
                ranking=ranking, k_optimal_value=k_opt, rCC_values=rCC, J_k_values=J_k, best_uncert_result=best)

## 6 · Validation, storage and report

`validation` holds out each experiment in turn, refits on the others and reports R² on the held-out data
(paper: 0.9985 ± 0.0015). `run_postprocessing` plots the t-values across rounds, the confidence regions and the rCC
curves, and writes Excel reports to `post_processing/`.

In [ ]:
from middoe.iden_valida import validation
from middoe.log_utils import save_to_jac, load_from_jac
from middoe.iden_utils import run_postprocessing

validres = validation(system, models, iden_opt, round_data)
print('validation R2:', validres['R2_stats']['M']['validation_mean'], '+/-', validres['R2_stats']['M']['validation_std'])

save_to_jac(round_data, purpose='iden')
run_postprocessing(round_data=load_from_jac()['iden'], solvers=['M'], selected_rounds=[1, 2, 3, 4],
                   plot_global_p_and_t=True, plot_confidence_spaces=True, plot_p_and_t_tests=True,
                   export_excel_reports=True, plot_estimability=True)